# Regulatory Compliance RAG Playground

This interactive notebook demonstrates the core components of the **Regulatory Compliance Intelligence Platform** step-by-step.

### Platform Core Blocks:
1. **PDF Ingestion & Parsing** (using `pypdf`)
2. **Metadata-Preserving Chunking** (using `RecursiveCharacterTextSplitter`)
3. **Embedding Vectorization** (using local `sentence-transformers`)
4. **ChromaDB Collection Persistence**
5. **Hybrid Retrieval Reranking** (80% Semantic + 20% Keyword Overlap)
6. **Evaluation Metrics** (Precision, Recall, Hit Rate)
7. **Compliance Risk Scoring** (LOW / MEDIUM / HIGH)
8. **Executive Report Exports** (Markdown & PDF)

### Step 1: Import Backend Modules

In [ ]:
import os
import sys

# Add root path to imports
sys.path.append(os.path.abspath(os.getcwd()))

from src.loader import DocumentLoader
from src.chunker import DocumentChunker
from src.embeddings import EmbeddingFactory
from src.vector_store import ChromaVectorStore
from src.retrieval import ComplianceRetriever
from src.risk_scoring import RiskScorer
from src.evaluation import RetrievalEvaluator
from src.report_generator import ReportGenerator

### Step 2: Initialize Mock PDF Data
We will programmatically write a sample compliance document to run our test inquiries.

In [ ]:
from fpdf import FPDF

os.makedirs("data", exist_ok=True)
pdf_path = "data/sample_audit_policy.pdf"

pdf = FPDF()
pdf.add_page()
pdf.set_font("helvetica", "B", 16)
pdf.multi_cell(pdf.epw, 10, "SAMPLE BANK DATA RETENTION POLICY", align="C")
pdf.ln(5)
pdf.set_font("helvetica", "", 11)
pdf.multi_cell(pdf.epw, 6, (
    "Section 1: Data Retention Rules.\n"
    "All customer transaction records must be retained for exactly 7 years after account closure.\n"
    "This policy follows regulatory advisory guidance and audit recommendations.\n\n"
    "Section 2: Suspicious Activity Review.\n"
    "Any suspected transaction related to money laundering or insider fraud must be escalated.\n"
    "A breach of these systems will result in immediate penalty and administrative sanction.\n"
    "A review exception will trigger an audit finding for escalation."
))
pdf.output(pdf_path)
print(f"Mock PDF created at: {pdf_path}")

### Step 3: Ingestion and Chunking

In [ ]:
loader = DocumentLoader()
pages = loader.load_pdf(pdf_path)

chunker = DocumentChunker(chunk_size=300, chunk_overlap=50)
chunks = chunker.split_pages(pages)

print(f"Parsed {len(pages)} pages.")
print(f"Generated {len(chunks)} overlapping text chunks.")
print("Sample Chunk Metadata:", chunks[0]["metadata"])

### Step 4: Vector Embeddings and Chroma DB Storage

In [ ]:
embedding_function = EmbeddingFactory.get_embedding_function(provider="local")
vector_store = ChromaVectorStore(persist_directory="vectorstore_playground", collection_name="playground_collection")

# Clear and add chunks
vector_store.rebuild_index()
vector_store.add_chunks(chunks, embedding_function)
print("Index built and loaded in ChromaDB successfully.")

### Step 5: Hybrid Search Retrieval

In [ ]:
retriever = ComplianceRetriever(vector_store=vector_store, embedding_function=embedding_function)
query = "money laundering penalties and audit exception escalation"

results = retriever.retrieve(query, top_k=3, relevance_threshold=0.2, hybrid=True)
for i, r in enumerate(results):
    print(f"\n[Match #{i+1}] Score: {r['final_score']:.2%} (Semantic: {r['semantic_score']:.2%}, Keyword: {r['keyword_score']:.2%})")
    print(f"Source: {r['metadata']['filename']} | Page: {r['metadata']['page']}")
    print(f"Excerpt: \"{r['text'].strip()}\"")

### Step 6: Risk Scoring and Evaluation

In [ ]:
response_text = "Suspected money laundering must be escalated to trigger an audit finding and avoid penalties."

risk_data = RiskScorer.score_content(query, response_text)
eval_report = RetrievalEvaluator.generate_evaluation_report(
    query=query,
    retrieved_chunks=results,
    ground_truth_filenames=["sample_audit_policy.pdf"]
)

print("Assigned Risk Level:", risk_data["risk_level"])
print("Risk Explanation:", risk_data["explanation"])
print("\nSearch Evaluation Precision:", eval_report["metrics"]["precision"])
print("Search Evaluation Recall:", eval_report["metrics"]["recall"])

### Step 7: Report Generation
We will synthesize our session data into Markdown and PDF reports.

In [ ]:
report_data = ReportGenerator.generate_report(
    query=query,
    response=response_text,
    risk_data=risk_data,
    retrieved_chunks=results,
    eval_data=eval_report,
    model_name="local-all-MiniLM-L6-v2-mock"
)

md_report = ReportGenerator.export_markdown(report_data)
pdf_report = ReportGenerator.export_pdf(report_data)

print(f"Markdown report exported to: {md_report}")
print(f"PDF report exported to: {pdf_report}")